In [1]:
"""
Reflexion Agent: Complete Implementation
A self-improving agent that learns from mistakes through reflection.

This implementation includes:
- Three-tier memory system
- Error classification and recovery
- Structured reflection
- Multi-step task planning
- Real tool execution with retry logic
"""

import json
import time
import re
from typing import Dict, List, Any, Optional
from dataclasses import dataclass
from datetime import datetime


# ============================================================================
# MEMORY SYSTEM
# ============================================================================

class AgentMemory:
    """
    Three-tier memory system:
    - Working: Current task execution
    - Episodic: Past task attempts  
    - Semantic: General learnings/patterns
    """
    
    def __init__(self):
        self.working_memory = {
            'task': None,
            'plan': [],
            'current_step': 0,
            'trajectory': []
        }
        self.episodic_memory = []
        self.semantic_memory = {
            'successful_patterns': [],
            'known_failures': [],
            'tool_guidelines': {}
        }
    
    def add_to_trajectory(self, step_type: str, content: Any):
        """Add step to current trajectory."""
        self.working_memory['trajectory'].append({
            'step': self.working_memory['current_step'],
            'type': step_type,
            'content': content,
            'timestamp': datetime.now().isoformat()
        })
        self.working_memory['current_step'] += 1
    
    def store_episode(self, task: str, trajectory: List, success: bool, reflection: str):
        """Store completed episode."""
        episode = {
            'task': task,
            'trajectory': trajectory,
            'success': success,
            'reflection': reflection,
            'timestamp': datetime.now().isoformat()
        }
        self.episodic_memory.append(episode)
        
        # Keep only last 50 episodes
        if len(self.episodic_memory) > 50:
            self.episodic_memory = self.episodic_memory[-50:]
    
    def store_learning(self, learning_type: str, content: Dict):
        """Store general learning."""
        if learning_type == 'pattern':
            self.semantic_memory['successful_patterns'].append(content)
        elif learning_type == 'failure':
            self.semantic_memory['known_failures'].append(content)
        elif learning_type == 'tool_guideline':
            tool_name = content['tool']
            if tool_name not in self.semantic_memory['tool_guidelines']:
                self.semantic_memory['tool_guidelines'][tool_name] = []
            self.semantic_memory['tool_guidelines'][tool_name].append(content['guideline'])
    
    def retrieve_relevant(self, task: str) -> Dict:
        """Retrieve memories relevant to current task."""
        # Simple keyword-based retrieval
        task_words = set(task.lower().split())
        
        # Find similar episodes
        relevant_episodes = []
        for ep in self.episodic_memory:
            ep_words = set(ep['task'].lower().split())
            overlap = len(task_words & ep_words) / max(len(task_words | ep_words), 1)
            if overlap > 0.2:
                relevant_episodes.append(ep)
        
        return {
            'episodes': relevant_episodes[-3:],  # Last 3 similar
            'patterns': self.semantic_memory['successful_patterns'][-5:],
            'failures': self.semantic_memory['known_failures'][-5:],
            'tool_guidelines': self.semantic_memory['tool_guidelines']
        }
    
    def clear_working_memory(self):
        """Reset for new task."""
        self.working_memory = {
            'task': None,
            'plan': [],
            'current_step': 0,
            'trajectory': []
        }
    
    def get_summary(self) -> Dict:
        """Get memory statistics."""
        return {
            'episodic_count': len(self.episodic_memory),
            'successful_episodes': sum(1 for ep in self.episodic_memory if ep['success']),
            'patterns_learned': len(self.semantic_memory['successful_patterns']),
            'failures_recorded': len(self.semantic_memory['known_failures']),
            'tools_with_guidelines': len(self.semantic_memory['tool_guidelines'])
        }


# ============================================================================
# ERROR HANDLING
# ============================================================================

class ErrorHandler:
    """Classify and handle different error types."""
    
    ERROR_PATTERNS = {
        'invalid_tool': {
            'pattern': r'tool.*not (found|available|exist)',
            'recovery': 'list_available_tools',
            'severity': 'high'
        },
        'invalid_params': {
            'pattern': r'(invalid|missing|unexpected) (parameter|argument)',
            'recovery': 'show_parameter_schema',
            'severity': 'medium'
        },
        'api_error': {
            'pattern': r'(api|http).*error|status.*[4-5]\d\d',
            'recovery': 'retry_with_backoff',
            'severity': 'medium'
        },
        'timeout': {
            'pattern': r'timeout|timed out',
            'recovery': 'retry_with_longer_timeout',
            'severity': 'low'
        },
        'rate_limit': {
            'pattern': r'rate limit|too many request',
            'recovery': 'wait_and_retry',
            'severity': 'low'
        },
        'value_error': {
            'pattern': r'(value|type).*error|invalid.*value',
            'recovery': 'validate_input',
            'severity': 'medium'
        }
    }
    
    def classify(self, error_message: str) -> Dict:
        """Classify error type."""
        error_lower = error_message.lower()
        
        for error_type, info in self.ERROR_PATTERNS.items():
            if re.search(info['pattern'], error_lower):
                return {
                    'type': error_type,
                    'recovery': info['recovery'],
                    'severity': info['severity'],
                    'message': error_message
                }
        
        return {
            'type': 'unknown',
            'recovery': 'reflect_and_retry',
            'severity': 'high',
            'message': error_message
        }
    
    def suggest_recovery(self, error_info: Dict, context: Dict) -> Dict:
        """Suggest recovery action."""
        recovery = error_info['recovery']
        
        if recovery == 'list_available_tools':
            return {
                'action': 'inform',
                'message': f"Available tools: {', '.join(context.get('available_tools', []))}"
            }
        elif recovery == 'show_parameter_schema':
            tool = context.get('tool_name')
            schema = context.get('tool_schemas', {}).get(tool, {})
            return {
                'action': 'inform',
                'message': f"Expected parameters for {tool}: {json.dumps(schema, indent=2)}"
            }
        elif recovery in ['retry_with_backoff', 'retry_with_longer_timeout']:
            attempt = context.get('attempt', 0)
            wait = min(2 ** attempt, 60)
            return {
                'action': 'wait_and_retry',
                'wait_seconds': wait
            }
        elif recovery == 'wait_and_retry':
            return {
                'action': 'wait_and_retry',
                'wait_seconds': 60
            }
        else:
            return {
                'action': 'reflect',
                'message': 'Analyze the error and determine root cause'
            }


# ============================================================================
# TOOLS
# ============================================================================

def create_calculator_tool():
    """Simple calculator tool."""
    def calculate(expression: str) -> float:
        """Evaluate mathematical expression."""
        try:
            # Safe eval with limited scope
            allowed_names = {
                'abs': abs, 'round': round,
                'min': min, 'max': max,
                'sum': sum, 'pow': pow
            }
            result = eval(expression, {"__builtins__": {}}, allowed_names)
            return float(result)
        except ZeroDivisionError:
            raise ValueError("Cannot divide by zero")
        except Exception as e:
            raise ValueError(f"Invalid expression: {str(e)}")
    
    return {
        'name': 'calculate',
        'description': 'Perform mathematical calculations',
        'function': calculate,
        'parameters': {
            'expression': 'str - Mathematical expression (e.g., "2 + 2", "10 * 5")'
        },
        'examples': [
            {'input': {'expression': '15 + 20'}, 'output': 35.0},
            {'input': {'expression': '100 / 4'}, 'output': 25.0}
        ]
    }


def create_search_tool():
    """Mock search tool."""
    def search(query: str) -> str:
        """Search for information (mock)."""
        # In real implementation, this would call actual search API
        mock_results = {
            'weather': 'Current weather: 72°F, partly cloudy',
            'python': 'Python is a high-level programming language...',
            'default': f'Search results for "{query}": [Mock search results]'
        }
        
        query_lower = query.lower()
        for key, result in mock_results.items():
            if key in query_lower:
                return result
        
        return mock_results['default']
    
    return {
        'name': 'search',
        'description': 'Search for information on the web',
        'function': search,
        'parameters': {
            'query': 'str - Search query'
        },
        'examples': [
            {'input': {'query': 'weather in Seattle'}, 'output': 'Current weather: ...'}
        ]
    }


def create_counter_tool():
    """Tool that counts up (for testing multi-step tasks)."""
    counter = {'value': 0}
    
    def increment(amount: int = 1) -> Dict:
        """Increment counter."""
        counter['value'] += amount
        return {
            'current_value': counter['value'],
            'message': f'Counter is now at {counter["value"]}'
        }
    
    def get_value() -> int:
        """Get current counter value."""
        return counter['value']
    
    def reset() -> Dict:
        """Reset counter."""
        counter['value'] = 0
        return {'message': 'Counter reset to 0'}
    
    return {
        'name': 'counter',
        'description': 'Increment, get, or reset a counter',
        'function': lambda action, **kwargs: {
            'increment': increment,
            'get': get_value,
            'reset': reset
        }[action](**kwargs),
        'parameters': {
            'action': "str - 'increment', 'get', or 'reset'",
            'amount': 'int - Amount to increment (only for increment action)'
        },
        'examples': [
            {'input': {'action': 'increment', 'amount': 5}, 'output': {'current_value': 5}},
            {'input': {'action': 'get'}, 'output': 5}
        ]
    }


# ============================================================================
# REFLEXION AGENT
# ============================================================================

class ReflexionAgent:
    """
    Self-improving agent with reflection loop.
    """
    
    def __init__(self, tools: List[Dict], llm_function):
        """
        Args:
            tools: List of tool definitions
            llm_function: Function that calls LLM (prompt -> response)
        """
        self.tools = {tool['name']: tool for tool in tools}
        self.llm = llm_function
        self.memory = AgentMemory()
        self.error_handler = ErrorHandler()
    
    def run(self, task: str, max_iterations: int = 5, verbose: bool = True) -> Optional[str]:
        """
        Execute task with reflection loop.
        
        Args:
            task: Task description
            max_iterations: Maximum attempts
            verbose: Print progress
            
        Returns:
            Final answer or None if failed
        """
        if verbose:
            print(f"\n{'='*70}")
            print(f"TASK: {task}")
            print(f"{'='*70}\n")
        
        self.memory.clear_working_memory()
        self.memory.working_memory['task'] = task
        
        # Retrieve relevant memories
        memory_context = self.memory.retrieve_relevant(task)
        
        if verbose and (memory_context['episodes'] or memory_context['failures']):
            print("📚 Found relevant past experiences:")
            for ep in memory_context['episodes']:
                status = "✓" if ep['success'] else "✗"
                print(f"  {status} {ep['task']}")
            print()
        
        # Main reflection loop
        for iteration in range(max_iterations):
            if verbose:
                print(f"\n{'─'*70}")
                print(f"Iteration {iteration + 1}/{max_iterations}")
                print(f"{'─'*70}")
            
            # Act
            result = self._act(task, memory_context, verbose)
            
            # Check if complete
            if result.get('done'):
                if verbose:
                    print(f"\n✓ Task completed successfully!")
                    print(f"Answer: {result['content']}")
                
                # Store successful episode
                self.memory.store_episode(
                    task=task,
                    trajectory=self.memory.working_memory['trajectory'],
                    success=True,
                    reflection="Successfully completed task"
                )
                
                return result['content']
            
            # Reflect on failure
            if result.get('error'):
                if verbose:
                    print(f"\n✗ Encountered error: {result['error_type']}")
                    print(f"Reflecting on failure...")
                
                reflection = self._reflect(result, memory_context, verbose)
                
                if verbose:
                    print(f"\n💡 Insight: {reflection['insight']}")
                
                # Store learning
                self.memory.store_learning('failure', reflection)
                
                # Update context
                memory_context = self.memory.retrieve_relevant(task)
        
        if verbose:
            print(f"\n✗ Max iterations reached. Task incomplete.")
        
        # Store failed episode
        self.memory.store_episode(
            task=task,
            trajectory=self.memory.working_memory['trajectory'],
            success=False,
            reflection="Max iterations exceeded"
        )
        
        return None
    
    def _act(self, task: str, memory_context: Dict, verbose: bool) -> Dict:
        """Execute next action."""
        # Build prompt
        prompt = self._build_prompt(task, memory_context)
        
        # Call LLM
        response = self.llm(prompt)
        
        if verbose:
            print(f"\n🤖 Agent response:")
            print(response[:300] + "..." if len(response) > 300 else response)
        
        # Parse response
        action = self._parse_action(response)
        
        if action is None:
            # Final answer
            answer = self._extract_final_answer(response)
            return {
                'done': True,
                'content': answer
            }
        
        # Execute tool
        try:
            if verbose:
                print(f"\n🔧 Executing: {action['tool']}({json.dumps(action['input'])})")
            
            result = self._execute_tool(action)
            
            if verbose:
                print(f"Result: {result}")
            
            self.memory.add_to_trajectory('action', action)
            self.memory.add_to_trajectory('observation', result)
            
            return {
                'done': False,
                'content': result
            }
            
        except Exception as e:
            error_info = self.error_handler.classify(str(e))
            
            self.memory.add_to_trajectory('error', {
                'action': action,
                'error': str(e),
                'type': error_info['type']
            })
            
            return {
                'done': False,
                'error': str(e),
                'error_type': error_info['type'],
                'action': action
            }
    
    def _reflect(self, failure_result: Dict, memory_context: Dict, verbose: bool) -> Dict:
        """Generate reflection on failure."""
        action = failure_result.get('action', {})
        error = failure_result.get('error', '')
        error_type = failure_result.get('error_type', 'unknown')
        
        # Build reflection prompt
        reflection_prompt = f"""Analyze this failure and provide a specific, actionable insight.

Action attempted:
Tool: {action.get('tool')}
Input: {json.dumps(action.get('input', {}))}

Error:
Type: {error_type}
Message: {error}

Recent actions:
{self._format_trajectory(self.memory.working_memory['trajectory'][-3:])}

Past failures to avoid:
{self._format_past_failures(memory_context['failures'])}

Provide:
1. Root cause (what specifically went wrong)
2. Actionable insight (how to avoid this in future)

Format:
Root Cause: [specific cause]
Insight: When [situation], always [action] because [reason]"""

        reflection_text = self.llm(reflection_prompt)
        
        if verbose:
            print(f"\n🔍 Reflection:")
            print(reflection_text)
        
        # Extract insight
        insight = self._extract_insight(reflection_text)
        
        return {
            'error': error,
            'error_type': error_type,
            'action': action,
            'full_reflection': reflection_text,
            'insight': insight
        }
    
    def _build_prompt(self, task: str, memory_context: Dict) -> str:
        """Build prompt for action selection."""
        # Tool descriptions
        tool_descriptions = []
        for name, tool in self.tools.items():
            desc = f"""
{name}:
  Description: {tool['description']}
  Parameters: {json.dumps(tool['parameters'], indent=4)}
  Examples: {json.dumps(tool['examples'][:2], indent=4)}
"""
            tool_descriptions.append(desc)
        
        # Memory context
        learnings = ""
        if memory_context['failures']:
            learnings += "\n⚠️  Past Failures to Avoid:\n"
            for failure in memory_context['failures'][-3:]:
                learnings += f"  - {failure.get('insight', failure.get('error', 'Unknown'))}\n"
        
        if memory_context['patterns']:
            learnings += "\n✓ Successful Patterns:\n"
            for pattern in memory_context['patterns'][-3:]:
                learnings += f"  - {pattern}\n"
        
        # Trajectory so far
        trajectory_text = ""
        if self.memory.working_memory['trajectory']:
            trajectory_text = "\nPrevious actions in this attempt:\n"
            trajectory_text += self._format_trajectory(
                self.memory.working_memory['trajectory'][-5:]
            )
        
        prompt = f"""You are a helpful agent that uses tools to accomplish tasks.

TASK: {task}

AVAILABLE TOOLS:
{chr(10).join(tool_descriptions)}

{learnings}
{trajectory_text}

INSTRUCTIONS:
1. Think step-by-step about what needs to be done
2. Use ONE tool at a time
3. Check tool parameters carefully against the schema
4. Learn from past failures shown above

RESPONSE FORMAT:
Thought: [your reasoning about what to do next]
Action: [tool_name]
Action Input: {{"param": "value"}}

OR if task is complete:
Thought: [summary of what was done]
Final Answer: [the result]

Your response:"""

        return prompt
    
    def _execute_tool(self, action: Dict) -> Any:
        """Execute tool call."""
        tool_name = action['tool']
        tool_input = action['input']
        
        if tool_name not in self.tools:
            raise ValueError(f"Tool '{tool_name}' not found. Available: {list(self.tools.keys())}")
        
        tool_func = self.tools[tool_name]['function']
        
        try:
            result = tool_func(**tool_input)
            return result
        except TypeError as e:
            # Parameter mismatch
            expected = self.tools[tool_name]['parameters']
            raise ValueError(f"Invalid parameters. Expected: {expected}. Error: {str(e)}")
    
    def _parse_action(self, response: str) -> Optional[Dict]:
        """Parse LLM response for action."""
        # Look for Action: and Action Input:
        action_match = re.search(r'Action:\s*(\w+)', response, re.IGNORECASE)
        input_match = re.search(r'Action Input:\s*({.*?})', response, re.DOTALL | re.IGNORECASE)
        
        if action_match and input_match:
            try:
                return {
                    'tool': action_match.group(1),
                    'input': json.loads(input_match.group(1))
                }
            except json.JSONDecodeError:
                return None
        
        return None
    
    def _extract_final_answer(self, response: str) -> str:
        """Extract final answer from response."""
        match = re.search(r'Final Answer:\s*(.+)', response, re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip()
        return response
    
    def _extract_insight(self, reflection: str) -> str:
        """Extract actionable insight from reflection."""
        match = re.search(r'Insight:\s*(.+)', reflection, re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip()
        
        # Fallback: return full reflection
        return reflection[:200]
    
    def _format_trajectory(self, trajectory: List[Dict]) -> str:
        """Format trajectory for display."""
        lines = []
        for step in trajectory:
            step_type = step['type']
            content = step['content']
            
            if step_type == 'action':
                lines.append(f"  Action: {content['tool']}({json.dumps(content['input'])})")
            elif step_type == 'observation':
                lines.append(f"  Result: {str(content)[:100]}")
            elif step_type == 'error':
                lines.append(f"  Error: {content.get('error', 'Unknown')}")
        
        return '\n'.join(lines) if lines else "  [No previous actions]"
    
    def _format_past_failures(self, failures: List[Dict]) -> str:
        """Format past failures for display."""
        if not failures:
            return "  [No past failures]"
        
        lines = []
        for f in failures[-3:]:
            insight = f.get('insight', f.get('error', 'Unknown'))
            lines.append(f"  - {insight[:150]}")
        
        return '\n'.join(lines)
    
    def get_memory_summary(self) -> Dict:
        """Get memory statistics."""
        return self.memory.get_summary()


# ============================================================================
# LLM INTERFACE (Mock for Demo)
# ============================================================================

def create_mock_llm():
    """Create mock LLM for demonstration."""
    
    def mock_llm(prompt: str) -> str:
        """
        Mock LLM that handles specific patterns.
        In real implementation, call actual LLM API (OpenAI, Anthropic, etc.)
        """
        prompt_lower = prompt.lower()
        
        # Calculate task
        if 'calculate' in prompt_lower and any(op in prompt_lower for op in ['15 * 20', '15*20', 'fifteen', 'twenty']):
            return """Thought: I need to multiply 15 by 20 to get the answer.
Action: calculate
Action Input: {"expression": "15 * 20"}"""
        
        # After calculation result
        if '300' in prompt_lower and 'calculate' in prompt_lower:
            return """Thought: I have calculated 15 * 20 = 300. The task is complete.
Final Answer: 300"""
        
        # Counter task
        if 'counter' in prompt_lower and 'increment' in prompt_lower:
            if '5' in prompt_lower or 'five' in prompt_lower:
                return """Thought: I need to increment the counter by 5.
Action: counter
Action Input: {"action": "increment", "amount": 5}"""
            return """Thought: I'll increment the counter.
Action: counter
Action Input: {"action": "increment", "amount": 1}"""
        
        # Reflection on divide by zero
        if 'root cause' in prompt_lower and 'zero' in prompt_lower:
            return """Root Cause: Attempted to divide by zero in mathematical expression
Insight: When using calculate tool, always validate that divisor is not zero before execution"""
        
        # Default
        return """Thought: Let me analyze what needs to be done.
Final Answer: Task requires more specific information to proceed."""
    
    return mock_llm


# ============================================================================
# DEMO
# ============================================================================

def demo_basic_task():
    """Demo: Simple calculation task."""
    print("\n" + "="*70)
    print("DEMO 1: Basic Task - Calculation")
    print("="*70)
    
    tools = [
        create_calculator_tool(),
        create_search_tool()
    ]
    
    llm = create_mock_llm()
    agent = ReflexionAgent(tools, llm)
    
    result = agent.run("What is 15 multiplied by 20?", max_iterations=3)
    
    print(f"\n{'='*70}")
    print("MEMORY SUMMARY")
    print("="*70)
    summary = agent.get_memory_summary()
    for key, value in summary.items():
        print(f"  {key}: {value}")


def demo_learning_from_failure():
    """Demo: Agent learns from errors."""
    print("\n" + "="*70)
    print("DEMO 2: Learning from Failure")
    print("="*70)
    
    tools = [create_calculator_tool()]
    llm = create_mock_llm()
    agent = ReflexionAgent(tools, llm)
    
    # First attempt will fail (divide by zero)
    print("\n[Attempt 1: This will fail due to divide by zero]")
    
    # Manually inject a failure
    try:
        agent.tools['calculate']['function']("10 / 0")
    except Exception as e:
        agent.memory.add_to_trajectory('error', {
            'action': {'tool': 'calculate', 'input': {'expression': '10 / 0'}},
            'error': str(e)
        })
        
        # Trigger reflection
        reflection = agent._reflect(
            {'error': str(e), 'error_type': 'value_error', 'action': {'tool': 'calculate'}},
            {'failures': [], 'patterns': []},
            verbose=True
        )
        
        agent.memory.store_learning('failure', reflection)
    
    print("\n[Now agent has learned about divide by zero]")
    print(f"Failures stored: {len(agent.memory.semantic_memory['known_failures'])}")


def main():
    """Run all demos."""
    print("\n" + "="*70)
    print("REFLEXION AGENT: SELF-IMPROVING THROUGH REFLECTION")
    print("="*70)
    
    demo_basic_task()
    demo_learning_from_failure()
    
    print("\n" + "="*70)
    print("KEY TAKEAWAYS")
    print("="*70)
    print("""
✓ Agent maintains 3-tier memory (working, episodic, semantic)
✓ Errors are classified and handled intelligently
✓ Reflection generates actionable insights
✓ Past failures guide future actions
✓ Agent improves with each attempt

NEXT STEPS:
1. Replace mock_llm with real LLM (OpenAI, Anthropic, etc.)
2. Add more sophisticated tools
3. Implement vector-based memory retrieval
4. Add multi-agent collaboration
5. Deploy for production use cases
    """)


if __name__ == "__main__":
    main()


REFLEXION AGENT: SELF-IMPROVING THROUGH REFLECTION

DEMO 1: Basic Task - Calculation

TASK: What is 15 multiplied by 20?


──────────────────────────────────────────────────────────────────────
Iteration 1/3
──────────────────────────────────────────────────────────────────────

🤖 Agent response:
Thought: Let me analyze what needs to be done.
Final Answer: Task requires more specific information to proceed.

✓ Task completed successfully!
Answer: Task requires more specific information to proceed.

MEMORY SUMMARY
  episodic_count: 1
  successful_episodes: 1
  patterns_learned: 0
  failures_recorded: 0
  tools_with_guidelines: 0

DEMO 2: Learning from Failure

[Attempt 1: This will fail due to divide by zero]

🔍 Reflection:
Root Cause: Attempted to divide by zero in mathematical expression
Insight: When using calculate tool, always validate that divisor is not zero before execution

[Now agent has learned about divide by zero]
Failures stored: 1

KEY TAKEAWAYS

✓ Agent maintains 3-tier